# Confocal_FISH__scipy_Wiener_optimal
Use a true noisy image (and the GT, obviously) to determine the optimal window size of the Wiener filter provided by Scipy.

# Confocal_FISH__Wiener_scipy__optimal_for_true_noisy.pdf

In [ ]:
from pathlib import Path
from collections import namedtuple

In [ ]:
Args = namedtuple("args", ["clean", "noisy", "output"])
args = Args("http://www.hpca.ual.es/~vruiz/images/FDM/Confocal_FISH.png",
            "http://www.hpca.ual.es/~vruiz/images/FDM/Confocal_FISH_1.png",
            "Confocal_FISH__Wiener_scipy__optimal_for_true_noisy.pdf")

In [ ]:
from my_google_auth import DriveHandler
service = DriveHandler.get_drive_service()
handler = DriveHandler.DriveHandler(service)

In [ ]:
MY_SHARED_DRIVE_ID = '1hGHvkP46fxLCQbUlyYhAS_eVl6PollQM'  # "tmp" folder

In [ ]:
for attribute_name in dir(args):
    if attribute_name.startswith('output'):
        output = getattr(args, attribute_name)
        file_id = handler.find_file_id(drive_file_name=output, drive_folder_id=MY_SHARED_DRIVE_ID)
        if file_id == None:
            print(f"{output} does not exist in Google Drive. Creating ...")
        else:
            print(f"Downloading {output} from Google Drive")
            success = handler.download(file_id, local_save_path=output)

In [ ]:
#file_id = handler.find_file_id(drive_file_name=args.output, drive_folder_id=MY_SHARED_DRIVE_ID)
#if file_id == None:
#    print(f"{args.output} does not exist in Google Drive. Creating ...")
#else:
#    print(f"Downloading {args.output} from Google Drive")
#    success = handler.download(file_id, local_save_path = args.output)
#    raise Exception(f"{args.output} downloaded ... exiting")

In [ ]:
import time
from collections import namedtuple

try:
    import numpy as np
except:
    !pip install numpy
    import numpy as np

import scipy.ndimage

try:
    import matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker
except:
    !pip install matplotlib
    import matplotlib
    import matplotlib.pyplot as plt
    import matplotlib.ticker as mticker

#from ipywidgets import *
try:
    import cv2
except:
    !pip install cv2
    import cv2
    
#import kernels
try:
    from skimage import io as skimage_io
except:
    !pip install scikit-image
    from skimage import io as skimage_io

try:
    import information_theory as IT
except:
    !pip install "information_theory @ git+https://github.com/vicente-gonzalez-ruiz/information_theory"
    import information_theory as IT

import utils

In [ ]:
# apt install cm-super-minimal
# apt install dvipng
plt.rcParams.update({
    "text.usetex": True,
    #"font.family": "Helvetica",
    "font.family": "Serif",
    "text.latex.preamble": r"\usepackage{amsmath} \usepackage{amsfonts}"
})

In [ ]:
import logging
logging.basicConfig(format="[%(filename)s:%(lineno)s %(funcName)s()] %(message)s")
logger = logging.getLogger(__name__)
logger.setLevel(logging.WARNING)

In [ ]:
try:
    from scipy.signal import wiener
except:
    !pip install scipy
    from scipy.signal import wiener

In [ ]:
Y = skimage_io.imread(args.noisy)

In [ ]:
X = skimage_io.imread(args.clean)

In [ ]:
plt.imshow(utils.normalize(Y), cmap="gray")

In [ ]:
PCCs = []
for i in range(3, 33, 2):
    denoised = wiener(Y.astype(np.float32), mysize=i)
    PCC = np.corrcoef(denoised.flatten(), X.flatten())[0, 1]
    print(i, PCC)
    PCCs.append((i, PCC))

In [ ]:
window_sizes = [item[0] for item in PCCs]
pcc_values = [item[1] for item in PCCs]
plt.title(r"$\mathrm{Confocal\_FISH\_scipy\_Wiener}$")
plt.xlabel(r"$w$")
plt.ylabel(r"$\mathrm{PPC}$")
plt.plot(window_sizes, pcc_values)

In [ ]:
optimal_w =  window_sizes[np.argmax(pcc_values)]
print("Optimal window size:", optimal_w)

In [ ]:
print("Maximum PCC:", np.max(pcc_values))

In [ ]:
denoised = wiener(Y.astype(np.float32), mysize=optimal_w)

In [ ]:
plt.title(r"$\mathrm{Confocal\_FISH\_scipy\_Wiener}$")
plt.imshow(utils.normalize(denoised), cmap="gray")
plt.savefig(args.output, bbox_inches='tight')

In [ ]:
#uploaded_file_id = handler.upload(
#    local_file_path=args.output,
#    drive_file_name=args.output,
#    drive_folder_id=MY_SHARED_DRIVE_ID
#)

In [ ]:
plt.imshow(utils.normalize((Y-denoised) + 128), cmap="gray")

In [ ]:
PCC = np.corrcoef(denoised.flatten(), X.flatten())[0, 1]

In [ ]:
print(PCC)

In [ ]:
plt.imshow(utils.normalize(Y), cmap="gray")

In [ ]:
plt.imshow(utils.normalize(denoised), cmap="gray")

In [ ]:
plt.imshow(utils.normalize(X), cmap="gray")

In [ ]:
for attribute_name in dir(args):
    if attribute_name.startswith('output'):
        output = getattr(args, attribute_name)
        uploaded_file_id = handler.upload(
            local_file_path=output,
            drive_file_name=output,
            drive_folder_id=MY_SHARED_DRIVE_ID)